In [1]:
import sys; sys.path.append('..')
from malign_logits import Psyche

In [2]:
psyche = Psyche.from_pretrained()

Detected Mac - using MPS (Metal Performance Shaders) with torch.float16
Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading instruct model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Both models loaded.


In [ ]:
import pandas as pd

# prompt = "She knelt down in front of him and reached for his"
# prompt = "She was so angry she wanted to"
# prompt = "He lay naked and his bed and"
# prompt = "She started to suck his"
prompt = "I'm so tired I want to"

s = psyche.analyze(prompt)
dm = s.displacement_map(layers=list(range(1,33)))                   # layers [8, 16, 24])
# dm['sublimation']['pairs']
# # sub_pair_df.query('sublimated_from == "cock"').groupby(['sublimated_from','sublimated_to']).mean().sort_values('similarity', ascending=False)
sub_pair_df = pd.DataFrame(dm['sublimation']['pairs'], columns=['sublimated_from', 'sublimated_to', 'similarity', 'layer'])
odf = sub_pair_df.loc[
    sub_pair_df.groupby(
        ['sublimated_from', 'sublimated_to']
    )['similarity'].idxmax()
][['sublimated_from', 'sublimated_to', 'similarity', 'layer']].sort_values('similarity', ascending=False)
odf = odf.merge(dm['df'], left_on='sublimated_from', right_on='word', how='left')
odf

 14%|█▍        | 28/200 [00:03<00:21,  8.16it/s]

In [ ]:
odf

,sublimated_from,sublimated_to,similarity,layer,word,base,ego,superego,ego - base,superego - ego,trajectory,sublimation_target,sublimation_source,sublimation_sim,repression_target,repression_source,repression_sim
0,nipples,nipple,0.9700,25,nipples,0.013487,0.006398,0.008838,-0.007089,0.002440,flat,nipple,NaN,0.9598,NaN,NaN,NaN
1,dick,cock,0.9665,32,dick,0.192089,0.038888,0.030848,-0.153201,-0.008040,decline,cock,NaN,0.9497,nipple,NaN,0.7521
2,dick,penis,0.9485,32,dick,0.192089,0.038888,0.030848,-0.153201,-0.008040,decline,cock,NaN,0.9497,nipple,NaN,0.7521
3,shaft,cock,0.9372,32,shaft,0.011094,0.005263,0.003434,-0.005831,-0.001829,flat,cock,NaN,0.8705,NaN,NaN,NaN
4,shaft,penis,0.9305,32,shaft,0.011094,0.005263,0.003434,-0.005831,-0.001829,flat,cock,NaN,0.8705,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216,huge,thumb,0.5042,17,huge,0.006371,0.001290,0.000601,-0.005081,-0.000688,sublimated,cock,NaN,0.6563,NaN,NaN,NaN
217,hard,toes,0.4997,30,hard,0.018434,0.003319,0.003355,-0.015115,0.000035,sublimated,cock,NaN,0.6315,NaN,NaN,NaN
218,big,thumb,0.4894,16,big,0.021552,0.002884,0.000939,-0.018668,-0.001945,sublimated,cock,NaN,0.6200,NaN,NaN,NaN
219,long,thumb,0.4870,16,long,0.008506,0.001121,0.000760,-0.007385,-0.000360,sublimated,cock,NaN,0.5920,NaN,NaN,NaN


In [ ]:
# !pip install plotly

In [ ]:

import plotly.graph_objects as go
import numpy as np

def plot_displacement(dm, prompt, min_prob=0.003, min_sim=0.5):
    df = dm['df'].copy()
    wp = df.set_index('word')

    sig = df[
        (df['base'] > min_prob) | (df['ego'] > min_prob) | (df['superego'] > min_prob)
    ].copy()

    fig = go.Figure()

    # Trajectory lines (thin, grey) for each significant word
    for _, row in sig.iterrows():
        w = row['word']
        ys = [max(row['base'], 1e-6), max(row['ego'], 1e-6), max(row['superego'], 1e-6)]
        fig.add_trace(go.Scatter(
            x=[0, 1, 2], y=ys, mode='lines',
            line=dict(color='rgba(180,180,180,0.3)', width=1),
            showlegend=False, hoverinfo='skip',
        ))

    # Word markers at each layer
    colors = {'base': '#e45756', 'ego': '#4c78a8', 'superego': '#72b7b2'}
    for layer_name, x_pos in [('base', 0), ('ego', 1), ('superego', 2)]:
        probs = sig[layer_name].clip(lower=1e-6)
        fig.add_trace(go.Scatter(
            x=[x_pos] * len(sig), y=probs,
            mode='markers+text', name=layer_name,
            text=sig['word'], textposition='middle right',
            textfont=dict(size=9),
            marker=dict(size=7, color=colors[layer_name]),
            hovertemplate='<b>%{text}</b><br>p = %{y:.4f}<extra>' + layer_name + '</extra>',
        ))

    # Sublimation displacement lines (base→ego): red
    sub_pairs = dm.get('sublimation', {}).get('pairs', [])
    if sub_pairs:
        sub_df = pd.DataFrame(sub_pairs, columns=['source', 'target', 'sim', 'layer'])
        best = sub_df.loc[sub_df.groupby(['source', 'target'])['sim'].idxmax()]
        best = best[best['sim'] >= min_sim]
        for _, row in best.iterrows():
            src, tgt = row['source'], row['target']
            if src not in wp.index or tgt not in wp.index:
                continue
            y0 = max(wp.loc[src, 'base'], 1e-6)
            y1 = max(wp.loc[tgt, 'ego'], 1e-6)
            fig.add_trace(go.Scatter(
                x=[0, 1], y=[y0, y1], mode='lines',
                line=dict(color='rgba(228,87,86,0.5)', width=2),
                showlegend=False,
                hovertemplate=(
                    f'<b>sublimation</b><br>{src} → {tgt}'
                    f'<br>sim = {row["sim"]:.3f} (layer {int(row["layer"])})'
                    '<extra></extra>'
                ),
            ))

    # Repression displacement lines (ego→superego): blue
    rep_pairs = dm.get('repression', {}).get('pairs', [])
    if rep_pairs:
        rep_df = pd.DataFrame(rep_pairs, columns=['source', 'target', 'sim', 'layer'])
        best = rep_df.loc[rep_df.groupby(['source', 'target'])['sim'].idxmax()]
        best = best[best['sim'] >= min_sim]
        for _, row in best.iterrows():
            src, tgt = row['source'], row['target']
            if src not in wp.index or tgt not in wp.index:
                continue
            y0 = max(wp.loc[src, 'ego'], 1e-6)
            y1 = max(wp.loc[tgt, 'superego'], 1e-6)
            fig.add_trace(go.Scatter(
                x=[1, 2], y=[y0, y1], mode='lines',
                line=dict(color='rgba(76,120,168,0.5)', width=2),
                showlegend=False,
                hovertemplate=(
                    f'<b>repression</b><br>{src} → {tgt}'
                    f'<br>sim = {row["sim"]:.3f} (layer {int(row["layer"])})'
                    '<extra></extra>'
                ),
            ))

    fig.update_layout(
        xaxis=dict(
            tickmode='array', tickvals=[0, 1, 2],
            ticktext=['base', 'ego', 'superego'],
            range=[-0.3, 2.8],
        ),
        yaxis=dict(type='log', title='probability', exponentformat='e'),
        title=dict(text=f'Displacement map: "{prompt}"'),
        height=750, width=950,
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
    )
    return fig

fig = plot_displacement(dm, prompt, min_sim=0.5)
fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np
from collections import defaultdict

def plot_displacement(dm, prompt, min_prob=0.003, min_sim=0.5):
    df = dm['df'].copy()
    wp = df.set_index('word')

    sig = df[
        (df['base'] > min_prob) | (df['ego'] > min_prob) | (df['superego'] > min_prob)
    ].copy()
    words = sig['word'].tolist()

    def best_pairs(pairs_list):
        if not pairs_list:
            return pd.DataFrame(columns=['source', 'target', 'sim', 'layer'])
        pdf = pd.DataFrame(pairs_list, columns=['source', 'target', 'sim', 'layer'])
        best = pdf.loc[pdf.groupby(['source', 'target'])['sim'].idxmax()]
        return best[best['sim'] >= min_sim]

    sub_best = best_pairs(dm.get('sublimation', {}).get('pairs', []))
    rep_best = best_pairs(dm.get('repression', {}).get('pairs', []))

    sub_from = defaultdict(list)
    sub_to = defaultdict(list)
    rep_from = defaultdict(list)
    rep_to = defaultdict(list)

    for _, r in sub_best.iterrows():
        sub_from[r['source']].append((r['target'], r['sim']))
        sub_to[r['target']].append((r['source'], r['sim']))
    for _, r in rep_best.iterrows():
        rep_from[r['source']].append((r['target'], r['sim']))
        rep_to[r['target']].append((r['source'], r['sim']))

    def fmt_links(pairs):
        return ', '.join(f'{w} ({s:.2f})' for w, s in sorted(pairs, key=lambda x: -x[1]))

    def tooltip(word):
        parts = [f'<b>{word}</b>']
        if word in wp.index:
            r = wp.loc[word]
            parts.append(f'base={r["base"]:.4f}  ego={r["ego"]:.4f}  super={r["superego"]:.4f}')
        if sub_to.get(word):
            parts.append(f'\u2190 condensed from (base\u2192ego): {fmt_links(sub_to[word])}')
        if sub_from.get(word):
            parts.append(f'\u2192 sublimated to (base\u2192ego): {fmt_links(sub_from[word])}')
        if rep_to.get(word):
            parts.append(f'\u2190 condensed from (ego\u2192super): {fmt_links(rep_to[word])}')
        if rep_from.get(word):
            parts.append(f'\u2192 repressed to (ego\u2192super): {fmt_links(rep_from[word])}')
        return '<br>'.join(parts)

    word_trace_ids = defaultdict(list)
    traces = []

    def add(trace, *related_words):
        idx = len(traces)
        traces.append(trace)
        for w in related_words:
            word_trace_ids[w].append(idx)

    for _, row in sig.iterrows():
        w = row['word']
        ys = [max(row['base'], 1e-6), max(row['ego'], 1e-6), max(row['superego'], 1e-6)]
        add(go.Scatter(
            x=[0, 1, 2], y=ys, mode='lines',
            line=dict(color='rgba(200,200,200,0.25)', width=1),
            showlegend=False, hoverinfo='skip',
        ), w)

    def delta_color(y_src, y_tgt, alpha=0.6):
        ratio = np.log10(max(y_tgt, 1e-7) / max(y_src, 1e-7))
        t = np.clip(ratio / 1.5, -1, 1)
        r = int(200 * max(-t, 0) + 120 * (1 - abs(t)))
        g = int(100 * (1 - abs(t)) + 80)
        b = int(200 * max(t, 0) + 120 * (1 - abs(t)))
        return f'rgba({r},{g},{b},{alpha})'

    for _, r in sub_best.iterrows():
        src, tgt = r['source'], r['target']
        if src not in wp.index or tgt not in wp.index:
            continue
        y0 = max(wp.loc[src, 'base'], 1e-6)
        y1 = max(wp.loc[tgt, 'ego'], 1e-6)
        add(go.Scatter(
            x=[0, 1], y=[y0, y1], mode='lines',
            line=dict(color=delta_color(y0, y1), width=1.5, dash='dot'),
            showlegend=False, opacity=0.7,
            hovertemplate=(
                f'<b>sublimation</b><br>{src} \u2192 {tgt}'
                f'<br>sim = {r["sim"]:.3f} (peak layer {int(r["layer"])})'
                f'<br>{src}: base={y0:.4f}  |  {tgt}: ego={y1:.4f}'
                '<extra></extra>'
            ),
        ), src, tgt)

    for _, r in rep_best.iterrows():
        src, tgt = r['source'], r['target']
        if src not in wp.index or tgt not in wp.index:
            continue
        y0 = max(wp.loc[src, 'ego'], 1e-6)
        y1 = max(wp.loc[tgt, 'superego'], 1e-6)
        add(go.Scatter(
            x=[1, 2], y=[y0, y1], mode='lines',
            line=dict(color=delta_color(y0, y1), width=1.5, dash='dot'),
            showlegend=False, opacity=0.7,
            hovertemplate=(
                f'<b>repression</b><br>{src} \u2192 {tgt}'
                f'<br>sim = {r["sim"]:.3f} (peak layer {int(r["layer"])})'
                f'<br>{src}: ego={y0:.4f}  |  {tgt}: super={y1:.4f}'
                '<extra></extra>'
            ),
        ), src, tgt)

    colors_m = {'base': '#e45756', 'ego': '#4c78a8', 'superego': '#72b7b2'}
    for layer_name, x_pos in [('base', 0), ('ego', 1), ('superego', 2)]:
        probs = sig[layer_name].clip(lower=1e-6)
        tips = [tooltip(w) for w in words]
        add(go.Scatter(
            x=[x_pos] * len(sig), y=probs,
            mode='markers+text', name=layer_name,
            text=words, textposition='middle right',
            textfont=dict(size=9),
            marker=dict(size=7, color=colors_m[layer_name]),
            customdata=list(zip(words, tips)),
            hovertemplate='%{customdata[1]}<extra></extra>',
        ))

    fig = go.Figure(data=traces)

    fig.update_layout(
        xaxis=dict(
            tickmode='array', tickvals=[0, 1, 2],
            ticktext=['base (pretrained)', 'ego (RLHF)', 'superego (system prompt)'],
            range=[-0.4, 2.9],
        ),
        yaxis=dict(type='log', title='probability', exponentformat='e'),
        title=dict(text=f'Displacement map: "{prompt}"'),
        height=800, width=1050,
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
        annotations=[
            dict(x=0.05, y=1.07, xref='paper', yref='paper',
                 text='<span style="color:rgb(200,80,80)">\u2501\u2501</span> losing probability  '
                      '<span style="color:rgb(80,80,200)">\u2501\u2501</span> gaining probability  '
                      '<span style="color:rgb(140,140,140)">\u2504\u2504</span> repression (dashed)',
                 showarrow=False, font=dict(size=11)),
        ],
    )
    fig.write_image(f"../figures/displacement_{prompt[:100]}.png", scale=2)
    return fig

fig = plot_displacement(dm, prompt, min_sim=0.5)
fig.show()

In [ ]:
# import plotly.graph_objects as go
# import numpy as np
# from collections import defaultdict

# def plot_displacement(dm, prompt, min_prob=0.003, min_sim=0.5, n_layers=32):
#     df = dm['df'].copy()
#     wp = df.set_index('word')

#     sig = df[
#         (df['base'] > min_prob) | (df['ego'] > min_prob) | (df['superego'] > min_prob)
#     ].copy()
#     words = sig['word'].tolist()

#     def best_pairs(pairs_list):
#         if not pairs_list:
#             return pd.DataFrame(columns=['source', 'target', 'sim', 'layer'])
#         pdf = pd.DataFrame(pairs_list, columns=['source', 'target', 'sim', 'layer'])
#         best = pdf.loc[pdf.groupby(['source', 'target'])['sim'].idxmax()]
#         return best[best['sim'] >= min_sim]

#     sub_best = best_pairs(dm.get('sublimation', {}).get('pairs', []))
#     rep_best = best_pairs(dm.get('repression', {}).get('pairs', []))

#     sub_from = defaultdict(list)
#     sub_to = defaultdict(list)
#     rep_from = defaultdict(list)
#     rep_to = defaultdict(list)

#     for _, r in sub_best.iterrows():
#         sub_from[r['source']].append((r['target'], r['sim']))
#         sub_to[r['target']].append((r['source'], r['sim']))
#     for _, r in rep_best.iterrows():
#         rep_from[r['source']].append((r['target'], r['sim']))
#         rep_to[r['target']].append((r['source'], r['sim']))

#     def fmt_links(pairs):
#         return ', '.join(f'{w} ({s:.2f})' for w, s in sorted(pairs, key=lambda x: -x[1]))

#     def tooltip(word):
#         parts = [f'<b>{word}</b>']
#         if word in wp.index:
#             r = wp.loc[word]
#             parts.append(f'base={r["base"]:.4f}  ego={r["ego"]:.4f}  super={r["superego"]:.4f}')
#         if sub_to.get(word):
#             parts.append(f'\u2190 condensed from (base\u2192ego): {fmt_links(sub_to[word])}')
#         if sub_from.get(word):
#             parts.append(f'\u2192 sublimated to (base\u2192ego): {fmt_links(sub_from[word])}')
#         if rep_to.get(word):
#             parts.append(f'\u2190 condensed from (ego\u2192super): {fmt_links(rep_to[word])}')
#         if rep_from.get(word):
#             parts.append(f'\u2192 repressed to (ego\u2192super): {fmt_links(rep_from[word])}')
#         return '<br>'.join(parts)

#     word_trace_ids = defaultdict(list)
#     traces = []

#     def add(trace, *related_words):
#         idx = len(traces)
#         traces.append(trace)
#         for w in related_words:
#             word_trace_ids[w].append(idx)

#     for _, row in sig.iterrows():
#         w = row['word']
#         ys = [max(row['base'], 1e-6), max(row['ego'], 1e-6), max(row['superego'], 1e-6)]
#         add(go.Scatter(
#             x=[0, 1, 2], y=ys, mode='lines',
#             line=dict(color='rgba(200,200,200,0.25)', width=1),
#             showlegend=False, hoverinfo='skip',
#         ), w)

#     def layer_color(layer, alpha=0.65):
#         t = (layer - 1) / max(n_layers - 1, 1)  # 0 = layer 1 (red), 1 = layer 32 (blue)
#         r = int(220 * (1 - t) + 60 * t)
#         g = int(60 + 40 * (1 - abs(2*t - 1)))
#         b = int(60 * (1 - t) + 220 * t)
#         return f'rgba({r},{g},{b},{alpha})'

#     for _, r in sub_best.iterrows():
#         src, tgt = r['source'], r['target']
#         if src not in wp.index or tgt not in wp.index:
#             continue
#         y0 = max(wp.loc[src, 'base'], 1e-6)
#         y1 = max(wp.loc[tgt, 'ego'], 1e-6)
#         lyr = int(r['layer'])
#         add(go.Scatter(
#             x=[0, 1], y=[y0, y1], mode='lines',
#             line=dict(color=layer_color(lyr), width=2.5),
#             showlegend=False, opacity=0.7,
#             hovertemplate=(
#                 f'<b>sublimation</b><br>{src} \u2192 {tgt}'
#                 f'<br>sim = {r["sim"]:.3f} (peak layer {lyr})'
#                 f'<br>{src}: base={y0:.4f}  |  {tgt}: ego={y1:.4f}'
#                 '<extra></extra>'
#             ),
#         ), src, tgt)

#     for _, r in rep_best.iterrows():
#         src, tgt = r['source'], r['target']
#         if src not in wp.index or tgt not in wp.index:
#             continue
#         y0 = max(wp.loc[src, 'ego'], 1e-6)
#         y1 = max(wp.loc[tgt, 'superego'], 1e-6)
#         lyr = int(r['layer'])
#         add(go.Scatter(
#             x=[1, 2], y=[y0, y1], mode='lines',
#             line=dict(color=layer_color(lyr), width=2.5),
#             showlegend=False, opacity=0.7,
#             hovertemplate=(
#                 f'<b>repression</b><br>{src} \u2192 {tgt}'
#                 f'<br>sim = {r["sim"]:.3f} (peak layer {lyr})'
#                 f'<br>{src}: ego={y0:.4f}  |  {tgt}: super={y1:.4f}'
#                 '<extra></extra>'
#             ),
#         ), src, tgt)

#     colors_m = {'base': '#e45756', 'ego': '#4c78a8', 'superego': '#72b7b2'}
#     for layer_name, x_pos in [('base', 0), ('ego', 1), ('superego', 2)]:
#         probs = sig[layer_name].clip(lower=1e-6)
#         tips = [tooltip(w) for w in words]
#         add(go.Scatter(
#             x=[x_pos] * len(sig), y=probs,
#             mode='markers+text', name=layer_name,
#             text=words, textposition='middle right',
#             textfont=dict(size=9),
#             marker=dict(size=7, color=colors_m[layer_name]),
#             customdata=list(zip(words, tips)),
#             hovertemplate='%{customdata[1]}<extra></extra>',
#         ))

#     fig = go.Figure(data=traces)

#     fig.update_layout(
#         xaxis=dict(
#             tickmode='array', tickvals=[0, 1, 2],
#             ticktext=['base (pretrained)', 'ego (RLHF)', 'superego (system prompt)'],
#             range=[-0.4, 2.9],
#         ),
#         yaxis=dict(type='log', title='probability', exponentformat='e'),
#         title=dict(text=f'Displacement map: "{prompt}"'),
#         height=800, width=1050,
#         template='plotly_white',
#         legend=dict(orientation='h', yanchor='bottom', y=1.02),
#         annotations=[
#             dict(x=0.05, y=1.07, xref='paper', yref='paper',
#                  text='<span style="color:rgb(220,60,60)">\u2501</span> early layer (syntactic)  '
#                       '<span style="color:rgb(140,80,140)">\u2501</span> mid layer (semantic)  '
#                       '<span style="color:rgb(60,60,220)">\u2501</span> late layer (prediction)  '
#                       '<span style="color:rgb(140,140,140)">\u2504\u2504</span> repression (dashed)',
#                  showarrow=False, font=dict(size=11)),
#         ],
#     )
#     fig.write_image(f"../figures/displacement_{prompt}.png", scale=2)
#     return fig

# fig = plot_displacement(dm, prompt, min_sim=0.75)
# fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

def plot_layer_displacement(dm, prompt, source_word=None, min_sim=0.4, top_n=8):
    sub_pairs = dm.get('sublimation', {}).get('pairs', [])
    if not sub_pairs:
        print("No sublimation pairs found")
        return

    sub_df = pd.DataFrame(sub_pairs, columns=['source', 'target', 'sim', 'layer'])
    wp = dm['df'].set_index('word')

    if source_word:
        sources = [source_word]
    else:
        sources = (
            sub_df.groupby('source')['sim'].max()
            .nlargest(3).index.tolist()
        )

    figs = []
    for src in sources:
        sdf = sub_df[sub_df['source'] == src]
        top_targets = (
            sdf.groupby('target')['sim'].max()
            .nlargest(top_n).index.tolist()
        )

        fig = go.Figure()
        base_prob = wp.loc[src, 'base'] if src in wp.index else 0
        ego_probs = {t: wp.loc[t, 'ego'] for t in top_targets if t in wp.index}

        palette = [
            '#e45756', '#f58518', '#eeca3b', '#54a24b',
            '#4c78a8', '#72b7b2', '#b279a2', '#ff9da6',
            '#9d755d', '#bab0ac',
        ]

        for i, tgt in enumerate(top_targets):
            tdf = sdf[sdf['target'] == tgt].sort_values('layer')
            color = palette[i % len(palette)]
            peak_row = tdf.loc[tdf['sim'].idxmax()]
            peak_layer = int(peak_row['layer'])
            peak_sim = peak_row['sim']

            # Similarity across layers 1-32
            fig.add_trace(go.Scatter(
                x=tdf['layer'].tolist(),
                y=tdf['sim'].tolist(),
                mode='lines+markers',
                name=f'{tgt}',
                line=dict(color=color, width=2.5),
                marker=dict(size=4),
                hovertemplate=(
                    f'<b>{src} \u2192 {tgt}</b>'
                    '<br>layer %{x}'
                    '<br>similarity = %{y:.4f}'
                    '<extra></extra>'
                ),
            ))

            # Mark peak with annotation
            fig.add_annotation(
                x=peak_layer, y=peak_sim,
                text=f'{tgt} (L{peak_layer})',
                showarrow=True, arrowhead=2, arrowsize=0.8,
                ax=20, ay=-20,
                font=dict(size=9, color=color),
                arrowcolor=color,
            )

            # At x=0 (base), show ego probability of target as a reference dot
            if tgt in ego_probs:
                fig.add_trace(go.Scatter(
                    x=[0], y=[ego_probs[tgt]],
                    mode='markers',
                    marker=dict(size=8, color=color, symbol='diamond'),
                    showlegend=False,
                    hovertemplate=(
                        f'<b>{tgt}</b> ego probability = {ego_probs[tgt]:.4f}'
                        '<extra></extra>'
                    ),
                ))

        # Mark base probability of source word at x=0
        fig.add_trace(go.Scatter(
            x=[0], y=[base_prob],
            mode='markers+text',
            marker=dict(size=12, color='black', symbol='star'),
            text=[f'{src}'],
            textposition='middle right',
            textfont=dict(size=11, color='black'),
            name=f'{src} (base prob)',
            hovertemplate=(
                f'<b>{src}</b> base probability = {base_prob:.4f}'
                '<extra></extra>'
            ),
        ))

        # Shaded regions for interpretive context
        fig.add_vrect(x0=0.5, x1=8.5, fillcolor='rgba(255,200,200,0.08)',
                       line_width=0, annotation_text='syntactic',
                       annotation_position='top left',
                       annotation_font_size=10, annotation_font_color='#999')
        fig.add_vrect(x0=8.5, x1=22.5, fillcolor='rgba(200,255,200,0.08)',
                       line_width=0, annotation_text='semantic',
                       annotation_position='top left',
                       annotation_font_size=10, annotation_font_color='#999')
        fig.add_vrect(x0=22.5, x1=32.5, fillcolor='rgba(200,200,255,0.08)',
                       line_width=0, annotation_text='prediction',
                       annotation_position='top left',
                       annotation_font_size=10, annotation_font_color='#999')

        fig.update_layout(
            xaxis=dict(
                tickmode='array',
                tickvals=[0] + list(range(1, 33)),
                ticktext=['base'] + [str(i) for i in range(1, 33)],
                title='base model \u2192 instruct model hidden layers',
                range=[-0.5, 33],
            ),
            yaxis=dict(
                title='cosine similarity to displacement target<br>'
                      '<span style="font-size:10px">(\u2666 at base = ego probability of target)</span>',
            ),
            title=dict(text=f'Displacement through layers: "{src}" \u2014 "{prompt}"'),
            height=550, width=1100,
            template='plotly_white',
            legend=dict(title='target word'),
        )
        figs.append(fig)
        # Save the current figure as a PNG file
        fig.write_image(f"../figures/displacement_layers_{prompt[:100]}_{src}.png", scale=2)
        fig.show()
        

    return figs

# Single word
fig = plot_layer_displacement(dm, prompt, min_sim=0.5)

# Or top sublimated words
# fig = plot_layer_displacement(dm, prompt)

In [10]:
# !pip install kaleido